# Calibration

Reliability diagrams and temperature scaling for neural classifiers.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import train_test_split

from prob_ml.calibration.metrics import ece, reliability_bins
from prob_ml.calibration.temperature import fit_temperature_scaling
from prob_ml.data.classification_dgp import ClassificationDGPConfig, generate_classification_data
from prob_ml.models.trainer import TrainConfig, train_classifier

In [ ]:
data = generate_classification_data(ClassificationDGPConfig(n_samples=4000, label_noise=0.15, seed=42))
X_train, X_tmp, y_train, y_tmp = train_test_split(data.X, data.y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

model = train_classifier(X_train, y_train, TrainConfig(epochs=40, seed=42))
with torch.no_grad():
    logits_val = model(torch.as_tensor(X_val, dtype=torch.float32)).cpu().numpy()
    logits_test = model(torch.as_tensor(X_test, dtype=torch.float32)).cpu().numpy()

ts = fit_temperature_scaling(logits_val, y_val, logits_eval=logits_test, y_eval=y_test)
print(f'T={ts.temperature:.3f}, ECE {ts.ece_before:.3f} -> {ts.ece_after:.3f}')

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

probs_before = sigmoid(logits_test)
bins = reliability_bins(probs_before, y_test)
bins_cal = reliability_bins(ts.probs_calibrated, y_test)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
ax.plot(bins['confidence'], bins['accuracy'], 'o-', label='Before')
ax.plot(bins_cal['confidence'], bins_cal['accuracy'], 's-', label='After TS')
ax.set_xlabel('Confidence')
ax.set_ylabel('Accuracy')
ax.legend()
ax.set_title('Reliability diagram')
plt.show()